# MMAR Simulation — Google Colab

[mmar-sim](https://github.com/Rifur/mmar-sim) · multifractal price paths, GOF validation, risk scenarios.

## Quick start

1. **Runtime → Run all** (or run cells top to bottom)
2. In **Step 2**, change `TICKER` / `N_STEPS` / `N_SIMS`
3. Or use the **interactive form** in Step 3

Examples: `2330.TW`, `6919.TW`, `NVDA`, `0050.TW`

## Step 1 — Setup (run once)

Clones the repo from GitHub if needed and installs dependencies.

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/Rifur/mmar-sim.git"


def _ensure_repo() -> None:
    if os.path.isfile("real_fractal_sim.py"):
        return
    if os.path.isdir("mmar-sim") and os.path.isfile("mmar-sim/real_fractal_sim.py"):
        os.chdir("mmar-sim")
        return
    print("Cloning mmar-sim from GitHub...")
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL])
    os.chdir("mmar-sim")


_ensure_repo()
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "yfinance>=1.4", "matplotlib>=3.8", "pandas>=2.0",
    "scipy>=1.11", "curl-cffi>=0.15",
])
print(f"Ready: {os.getcwd()}")

## Step 2 — Edit & Run (simplest)

Change the three lines below, then run this cell.

In [ ]:
%matplotlib inline
from real_fractal_sim_colab import run_colab

TICKER = "6919.TW"   # 2330.TW, NVDA, 0050.TW ...
N_STEPS = 20         # simulation horizon (trading days)
N_SIMS = 10000       # number of Monte Carlo paths

result = run_colab(TICKER, n_steps=N_STEPS, n_sims=N_SIMS, seed=42)

print(f"\nGOF score: {result['fit']['score']:.0f} / 100")
print(f"Charts: {result['mmar_path']}  |  {result['gof_path']}")

## Step 3 — Interactive form (optional)

Type a ticker and click **Run MMAR**.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from real_fractal_sim_colab import run_colab

ticker_in = widgets.Text(value="2330.TW", description="Ticker", layout=widgets.Layout(width="260px"))
steps_in = widgets.IntSlider(value=252, min=5, max=504, step=1, description="Days", readout_format="d")
sims_in = widgets.IntSlider(value=5000, min=500, max=20000, step=500, description="Paths", readout_format="d")
run_btn = widgets.Button(description="Run MMAR", button_style="success", icon="play")
out = widgets.Output()


def _on_run(_):
    with out:
        clear_output(wait=True)
        r = run_colab(
            ticker_in.value.strip().upper(),
            n_steps=steps_in.value,
            n_sims=sims_in.value,
            seed=42,
        )
        globals()["result"] = r
        print(f"GOF score: {r['fit']['score']:.0f} / 100")
        print(f"Charts: {r['mmar_path']}  |  {r['gof_path']}")


run_btn.on_click(_on_run)
display(widgets.VBox([ticker_in, steps_in, sims_in, run_btn, out]))

## Step 4 — Download charts

In [ ]:
try:
    from google.colab import files
    files.download(result["mmar_path"])
    files.download(result["gof_path"])
except (ImportError, NameError, KeyError):
    print("Run Step 2 or Step 3 first.")

## One-liner (copy into any cell)

```python
from real_fractal_sim_colab import run_colab
result = run_colab("6919.TW", n_steps=20, n_sims=10000)
print(result["fit"]["score"])
```

More options: `hist_start="2020-01-01"`, `calibrate_body=False`, `stress_tail_weight=0.55`